# Day 39 — Backtracking and pruning: N-Queens and Sudoku

Backtracking is depth-first search over a tree of partial answers that is never
actually built. Place one piece, recurse, take the piece back, try the next one.
Three lines carry every problem in this notebook: **choose, explore, unchoose.**

Two things decide whether it works. **Pruning decides whether it finishes**: reject
a partial answer the moment it breaks a rule and the whole subtree below it is
never visited. **The unchoose step decides whether it is right**: backtracking bugs
almost never crash. Below, a missing undo reports that 8-queens has no solution, a
missing copy returns eight empty lists, and a symmetry shortcut is exact for every
even board size and wrong on every odd one that has a solution.

In [1]:
from __future__ import annotations
import math, sys, time
from itertools import permutations, product
sys.setrecursionlimit(10_000)

## 1. The template on the smallest problem: every subset of [1, 2, 3]

Each call records the path it arrived with, then tries to extend it with every
later number. `path.pop()` is the unchoose: it is what lets the loop try `[1, 3]`
after `[1, 2]`, because the 2 has to come off first.

In [2]:
def subsets(nums: list[int], trace: list | None = None) -> list[list[int]]:
    res: list[list[int]] = []
    path: list[int] = []

    def go(start: int) -> None:
        res.append(path[:])                  # record a COPY of the path
        if trace is not None:
            trace.append(path[:])
        for i in range(start, len(nums)):
            path.append(nums[i])             # choose
            go(i + 1)                        # explore
            path.pop()                       # unchoose

    go(0)
    return res

In [3]:
trace = []
subs = subsets([1, 2, 3], trace)
for p in trace:
    print('  ' * len(p) + str(p))
print(f'-> {len(subs)} subsets (2^3 = 8)')

[]
  [1]
    [1, 2]
      [1, 2, 3]
    [1, 3]
  [2]
    [2, 3]
  [3]
-> 8 subsets (2^3 = 8)


## 2. Silent failure #1: storing the path instead of a copy

Same function, `path[:]` written as `path`. The result has the right length, so a
test that only checks `len` passes. But all eight slots are the *same* list object,
and by the time the search returns every `pop()` has emptied it.

In [4]:
def subsets_aliased(nums: list[int]) -> list[list[int]]:
    """The same function with `path[:]` written as `path`."""
    res: list[list[int]] = []
    path: list[int] = []

    def go(start: int) -> None:
        res.append(path)                     # BUG: stores the list object itself
        for i in range(start, len(nums)):
            path.append(nums[i])
            go(i + 1)
            path.pop()

    go(0)
    return res

In [5]:
bad = subsets_aliased([1, 2, 3])
print('res.append(path) ->', bad)
print('len:', len(bad), '  distinct list objects:', len({id(x) for x in bad}))

res.append(path) -> [[], [], [], [], [], [], [], []]
len: 8   distinct list objects: 1


## 3. N-Queens: one queen per row, three sets for the attacks

Put one queen in each row, so the only decision at row `r` is the column. A square
`(r, c)` is attacked along its column `c`, along its "\\" diagonal (every square on
it has the same `r - c`) and along its "/" diagonal (same `r + c`). Three sets make
the safety check a constant-time lookup, and a failing check is the pruning: that
square's whole subtree is skipped.

In [6]:
def queens(n: int, stats: dict | None = None) -> list[list[int]]:
    """All solutions; each is a list of column indices, one per row."""
    cols: set[int] = set()
    diag: set[int] = set()       # r - c
    anti: set[int] = set()       # r + c
    board: list[int] = []
    sols: list[list[int]] = []
    st = stats if stats is not None else {}
    st.setdefault('nodes', 0)
    st.setdefault('tested', 0)

    def go(r: int) -> None:
        st['nodes'] += 1
        if r == n:
            sols.append(board[:])
            return
        for c in range(n):
            st['tested'] += 1
            if c in cols or r - c in diag or r + c in anti:
                continue                                  # prune
            cols.add(c); diag.add(r - c); anti.add(r + c); board.append(c)
            go(r + 1)
            board.pop(); cols.remove(c); diag.remove(r - c); anti.remove(r + c)

    go(0)
    return sols


def draw_board(cols: list[int]) -> list[str]:
    n = len(cols)
    return [' '.join('Q' if cols[r] == c else '.' for c in range(n)) for r in range(n)]

In [7]:
st = {}
sols = queens(8, st)
print(f'8 queens: {len(sols)} solutions. The first one found:')
for line in draw_board(sols[0]):
    print('   ', line)
print('columns by row:', sols[0])

8 queens: 92 solutions. The first one found:
    Q . . . . . . .
    . . . . Q . . .
    . . . . . . . Q
    . . . . . Q . .
    . . Q . . . . .
    . . . . . . Q .
    . Q . . . . . .
    . . . Q . . . .
columns by row: [0, 4, 7, 5, 2, 6, 1, 3]


## 4. What the pruning buys

Each rule we build into the structure of the search shrinks the space: choosing any
8 of the 64 squares, one queen per row, one per row *and* column. Diagonal pruning
cuts it further, down to a couple of thousand nodes. The honest comparison is
against the full column-permutation tree, which a search without diagonal pruning
would walk.

In [8]:
def queens_by_permutation(n: int) -> tuple[int, int]:
    """No pruning on diagonals: generate every column permutation and only
    check the diagonals once the board is full.  Returns (solutions, boards
    checked)."""
    found = checked = 0
    for p in permutations(range(n)):
        checked += 1
        if (len({r - c for r, c in enumerate(p)}) == n
                and len({r + c for r, c in enumerate(p)}) == n):
            found += 1
    return found, checked


def permutation_tree_nodes(n: int) -> int:
    """Nodes in the column-permutation tree (1 + n + n(n-1) + ... + n!)."""
    total, level = 1, 1
    for k in range(n):
        level *= n - k
        total += level
    return total

In [9]:
ladder = [('any 8 of the 64 squares', math.comb(64, 8)),
          ('one queen per row', 8 ** 8),
          ('one per row and per column (8!)', math.factorial(8)),
          ('backtracking with diagonal pruning', st['nodes'])]
for label, v in ladder:
    print(f'{label:<36} {v:>15,}')
hits, checked = queens_by_permutation(8)
print(f'generate-then-check: {checked:,} full boards for {hits} hits')
print(f'permutation tree: {permutation_tree_nodes(8):,} nodes = '
      f'{permutation_tree_nodes(8) / st["nodes"]:.0f}x the pruned search')
print()
print(' n  solutions     nodes')
for n in range(1, 11):
    s2 = {}
    k = len(queens(n, s2))
    print(f'{n:>2} {k:>9} {s2["nodes"]:>9,}')

any 8 of the 64 squares                4,426,165,368
one queen per row                         16,777,216
one per row and per column (8!)               40,320
backtracking with diagonal pruning             2,057
generate-then-check: 40,320 full boards for 92 hits
permutation tree: 109,601 nodes = 53x the pruned search

 n  solutions     nodes
 1         1         2
 2         0         3
 3         0         6
 4         2        17
 5        10        54
 6         4       153
 7        40       552
 8        92     2,057
 9       352     8,394
10       724    35,539


## 5. Silent failure #2: forgetting the unchoose

Keep `board.pop()`, delete the three `set.remove()` calls. Every queen that is
taken back leaves its column and diagonals marked, so these ghosts keep attacking
squares nobody occupies. The search dies after nine nodes and reports zero
solutions: it claims 8-queens is impossible, and nothing is raised.

In [10]:
def queens_no_undo(n: int) -> tuple[list[list[int]], int, list[list[int]]]:
    """Same search, but the unchoose line only pops the board and forgets
    to release the column and the two diagonals.  Returns (solutions,
    nodes visited, every partial board it entered)."""
    cols: set[int] = set(); diag: set[int] = set(); anti: set[int] = set()
    board: list[int] = []
    sols: list[list[int]] = []
    seen: list[list[int]] = []

    def go(r: int) -> None:
        seen.append(board[:])
        if r == n:
            sols.append(board[:])
            return
        for c in range(n):
            if c in cols or r - c in diag or r + c in anti:
                continue
            cols.add(c); diag.add(r - c); anti.add(r + c); board.append(c)
            go(r + 1)
            board.pop()                          # BUG: sets never shrink

    go(0)
    return sols, len(seen), seen

In [11]:
bad_sols, bad_nodes, seen = queens_no_undo(8)
print(f'{len(bad_sols)} solutions after {bad_nodes} nodes. Every partial board it entered:')
for b in seen:
    print('   ', b)

0 solutions after 9 nodes. Every partial board it entered:
    []
    [0]
    [0, 2]
    [0, 2, 4]
    [0, 2, 4, 1]
    [0, 2, 4, 1, 3]
    [0, 2, 4, 1, 7]
    [0, 2, 6]
    [5]


## 6. Pruning must never cut a real answer: the mirror-symmetry shortcut

Flipping a solution left to right gives another solution, so a tempting speed-up
is to only try the left half of row 0 and double the count. It is exact on every
even board. On an odd board the middle column is its own mirror image: the
shortcut never searches it and never counts it. (n = 3 comes out right only
because it has no solutions to lose.)

In [12]:
def queens_count_half(n: int) -> int:
    """A tempting symmetry shortcut: only try the left half of row 0 and
    double the count, since the mirror image of a solution is a solution.
    Exact whenever n is even.  For odd n it never tries the middle column."""
    cols: set[int] = set(); diag: set[int] = set(); anti: set[int] = set()
    count = 0

    def go(r: int) -> None:
        nonlocal count
        if r == n:
            count += 1
            return
        for c in (range(n // 2) if r == 0 else range(n)):
            if c in cols or r - c in diag or r + c in anti:
                continue
            cols.add(c); diag.add(r - c); anti.add(r + c)
            go(r + 1)
            cols.remove(c); diag.remove(r - c); anti.remove(r + c)

    go(0)
    return 2 * count


def queens_count_half_fixed(n: int) -> int:
    """The shortcut done right: the middle column of an odd board is its
    own mirror image, so it is searched once and counted once."""
    def from_first(cs) -> int:
        cols: set[int] = set(); diag: set[int] = set(); anti: set[int] = set()
        count = 0

        def go(r: int) -> None:
            nonlocal count
            if r == n:
                count += 1
                return
            for c in (cs if r == 0 else range(n)):
                if c in cols or r - c in diag or r + c in anti:
                    continue
                cols.add(c); diag.add(r - c); anti.add(r + c)
                go(r + 1)
                cols.remove(c); diag.remove(r - c); anti.remove(r + c)

        go(0)
        return count

    total = 2 * from_first(range(n // 2))
    if n % 2:
        total += from_first([n // 2])
    return total

In [13]:
print(' n  true  halved  fixed')
for n in range(1, 11):
    t, h, f = len(queens(n)), queens_count_half(n), queens_count_half_fixed(n)
    print(f'{n:>2} {t:>5} {h:>7} {f:>6}   {"" if t == h else "<- halved is wrong"}')

 n  true  halved  fixed
 1     1       0      1   <- halved is wrong
 2     0       0      0   
 3     0       0      0   
 4     2       2      2   
 5    10       8     10   <- halved is wrong
 6     4       4      4   
 7    40      34     40   <- halved is wrong
 8    92      92     92   
 9   352     298    352   <- halved is wrong
10   724     724    724   


## 7. The bitmask version (Day 37's trick): same tree, cheaper nodes

Keep the three attack sets as bit rows seen from the current row. Moving down one
row shifts the "\\" attacks one way and the "/" attacks the other, so no `r - c`
bookkeeping is needed; `free & -free` picks the lowest free column. The node count
is identical. Only the cost per node changes.

In [14]:
def queens_count_bits(n: int, stats: dict | None = None) -> int:
    """Day 37's bitmask trick applied to N-Queens (LeetCode 52).

    cols / ld / rd are bit rows of attacked columns as seen from the current
    row.  Moving down one row shifts the "\" attacks right and the "/"
    attacks left, so no r - c bookkeeping is needed at all."""
    full = (1 << n) - 1
    st = stats if stats is not None else {}
    st.setdefault('nodes', 0)

    def go(cols: int, ld: int, rd: int) -> int:
        st['nodes'] += 1
        if cols == full:
            return 1
        total = 0
        free = full & ~(cols | ld | rd)
        while free:
            bit = free & -free                  # lowest free column
            free ^= bit
            total += go(cols | bit, ((ld | bit) << 1) & full, (rd | bit) >> 1)
        return total

    return go(0, 0, 0)

In [15]:
def timed(fn, *a, **kw):
    t = time.perf_counter(); out = fn(*a, **kw)
    return out, time.perf_counter() - t

s_sets, s_bits = {}, {}
sol10, t_sets = timed(queens, 10, s_sets)
cnt10, t_bits = timed(queens_count_bits, 10, s_bits)
print(f'n = 10  sets: {len(sol10)} solutions, {s_sets["nodes"]:,} nodes')
print(f'        bits: {cnt10} solutions, {s_bits["nodes"]:,} nodes')
print(f'bits version is {t_sets / t_bits:.1f}x faster on this run')

n = 10  sets: 724 solutions, 35,539 nodes
        bits: 724 solutions, 35,539 nodes
bits version is 3.8x faster on this run


## 8. Sudoku: which cell you branch on beats how fast you check it

The textbook solver fills the first empty cell in reading order with digits 1 to 9.
MRV (*minimum remaining values*) instead branches on the empty cell with the
**fewest** legal digits. A cell with one candidate is filled with no branching; a
cell with zero candidates proves the current path is dead immediately, many levels
before reading order would discover it.

The third puzzle is a 17-clue board whose top row solves to `987654321`, the exact
reverse of the order in which the naive solver tries digits. Running naive to the
end there takes about two minutes of pure Python (69,175,317 nodes); the
notebook caps it and uses the measured count, which `python backtracking.py --full`
reproduces.

In [16]:
PUZZLES: dict[str, str] = {
    # the example board from LeetCode 37: 30 givens, most cells are forced
    'LeetCode 37': '53..7....6..195....98....6.8...6...34..8.3..17...2...6.'
                   '6....28....419..5....8..79',
    # Arto Inkala's 2012 "world's hardest sudoku": 21 givens
    'Inkala': '8..........36......7..9.2...5...7.......457.....1...3...1'
              '....68..85...1..9....4..',
    # a 17-given puzzle whose top row solves to 987654321 - the exact
    # reverse of the order in which a naive solver tries the digits
    'anti-brute': '..............3.85..1.2.......5.7.....4...1...9.......5'
                  '......73..2.1........4...9',
}


# Nodes the naive row-major solver visits on 'anti-brute'.  Measured by
# sudoku_rowmajor_bits(PUZZLES['anti-brute'], cap=None) - about two minutes of
# pure Python, so main() only reruns it with --full.
ANTI_BRUTE_NAIVE_NODES = 69_175_317


def parse(s: str) -> list[list[int]]:
    return [[0 if ch == '.' else int(ch) for ch in s[r * 9:r * 9 + 9]] for r in range(9)]


def box_of(r: int, c: int) -> int:
    return r // 3 * 3 + c // 3


def is_valid_solution(g: list[list[int]], givens: str) -> bool:
    want = set(range(1, 10))
    for i in range(9):
        if set(g[i]) != want or {g[r][i] for r in range(9)} != want:
            return False
    for b in range(9):
        if {g[r][c] for r in range(9) for c in range(9) if box_of(r, c) == b} != want:
            return False
    return all(ch == '.' or int(ch) == g[i // 9][i % 9] for i, ch in enumerate(givens))


class Cap(Exception):
    """Raised when a search exceeds its node budget."""


def sudoku_naive(s: str, cap: int | None = None):
    """The textbook solver: first empty cell in reading order, digits 1..9,
    and a safety test that scans the row, the column and the 3x3 box.
    Returns (grid or None, nodes visited); grid is None if the cap hit."""
    g = parse(s)
    nodes = 0

    def safe(r: int, c: int, v: int) -> bool:
        for i in range(9):
            if g[r][i] == v or g[i][c] == v:
                return False
        br, bc = r // 3 * 3, c // 3 * 3
        for i in range(br, br + 3):
            for j in range(bc, bc + 3):
                if g[i][j] == v:
                    return False
        return True

    def go() -> bool:
        nonlocal nodes
        nodes += 1
        if cap is not None and nodes > cap:
            raise Cap
        for r in range(9):
            for c in range(9):
                if g[r][c] == 0:
                    for v in range(1, 10):
                        if safe(r, c, v):
                            g[r][c] = v                  # choose
                            if go():                     # explore
                                return True
                            g[r][c] = 0                  # unchoose
                    return False                         # no digit fits here
        return True                                      # no empty cell left

    try:
        ok = go()
    except Cap:
        return None, nodes - 1
    return (g if ok else None), nodes


def sudoku_rowmajor_bits(s: str, cap: int | None = None):
    """Same order as sudoku_naive - so exactly the same tree and the same
    node count - but the row / column / box tests are three bitmasks
    (bit v set = digit v already used) instead of 27 cell reads."""
    g = parse(s)
    rows = [0] * 9; cols = [0] * 9; boxes = [0] * 9
    for r in range(9):
        for c in range(9):
            if g[r][c]:
                bit = 1 << g[r][c]
                rows[r] |= bit; cols[c] |= bit; boxes[box_of(r, c)] |= bit
    empty = [(r, c) for r in range(9) for c in range(9) if g[r][c] == 0]
    nodes = 0

    def go(k: int) -> bool:
        nonlocal nodes
        nodes += 1
        if cap is not None and nodes > cap:
            raise Cap
        if k == len(empty):
            return True
        r, c = empty[k]
        b = box_of(r, c)
        free = ~(rows[r] | cols[c] | boxes[b]) & 0x3FE      # bits 1..9
        while free:
            bit = free & -free                              # smallest digit first
            free ^= bit
            rows[r] |= bit; cols[c] |= bit; boxes[b] |= bit
            g[r][c] = bit.bit_length() - 1
            if go(k + 1):
                return True
            rows[r] ^= bit; cols[c] ^= bit; boxes[b] ^= bit
            g[r][c] = 0
        return False

    try:
        ok = go(0)
    except Cap:
        return None, nodes - 1
    return (g if ok else None), nodes


def sudoku_mrv(s: str, first_pick: list | None = None):
    """Minimum remaining values: branch on the empty cell that has the FEWEST
    legal digits.  A cell with one candidate is filled with no branching at
    all; a cell with zero candidates proves the current path is dead right
    now, many levels before the naive order would find out."""
    g = parse(s)
    rows = [0] * 9; cols = [0] * 9; boxes = [0] * 9
    empty: list[tuple[int, int]] = []
    for r in range(9):
        for c in range(9):
            if g[r][c]:
                bit = 1 << g[r][c]
                rows[r] |= bit; cols[c] |= bit; boxes[box_of(r, c)] |= bit
            else:
                empty.append((r, c))
    nodes = 0

    def go() -> bool:
        nonlocal nodes
        nodes += 1
        if not empty:
            return True
        best, best_free, best_k = -1, 0, 10
        for i, (r, c) in enumerate(empty):
            free = ~(rows[r] | cols[c] | boxes[box_of(r, c)]) & 0x3FE
            k = free.bit_count() if hasattr(int, 'bit_count') else bin(free).count('1')
            if k < best_k:
                best, best_free, best_k = i, free, k
                if k <= 1:
                    break                       # cannot do better than forced
        if best_k == 0:
            return False                        # a cell with no legal digit
        r, c = empty[best]
        if first_pick is not None and not first_pick:
            first_pick.append(((r, c), best_k))
        empty[best] = empty[-1]; empty.pop()    # O(1) removal
        b = box_of(r, c)
        free = best_free
        while free:
            bit = free & -free
            free ^= bit
            rows[r] |= bit; cols[c] |= bit; boxes[b] |= bit
            g[r][c] = bit.bit_length() - 1
            if go():
                return True
            rows[r] ^= bit; cols[c] ^= bit; boxes[b] ^= bit
            g[r][c] = 0
        empty.append((r, c))                    # put the cell back where it was
        empty[best], empty[-1] = empty[-1], empty[best]
        return False

    ok = go()
    return (g if ok else None), nodes

In [17]:
NAIVE_CAP = 200_000
print(f'{"puzzle":<12}{"givens":>7}{"naive nodes":>14}{"MRV nodes":>11}{"ratio":>9}')
for name, s in PUZZLES.items():
    g1, n1 = sudoku_naive(s, cap=NAIVE_CAP)
    g2, n2 = sudoku_mrv(s)
    assert is_valid_solution(g2, s)
    n1 = n1 if g1 is not None else ANTI_BRUTE_NAIVE_NODES
    print(f'{name:<12}{sum(ch != "." for ch in s):>7}{n1:>14,}{n2:>11,}{n1 / n2:>8,.0f}x')

pick = []
ab, _ = sudoku_mrv(PUZZLES['anti-brute'], pick)
(r, c), k = pick[0]
print(f'anti-brute top row: {"".join(map(str, ab[0]))};  MRV first branches on '
      f'row {r}, col {c} ({k} candidates)')

puzzle       givens   naive nodes  MRV nodes    ratio
LeetCode 37      30         4,209         52      81x
Inkala           21        49,559     15,573       3x
anti-brute       17    69,175,317     17,918   3,861x
anti-brute top row: 987654321;  MRV first branches on row 1, col 2 (3 candidates)


Fewer nodes is not automatically less time. MRV scans every empty cell at every
node, so on a puzzle where it only saves a factor of three it can lose on the
clock to a plain reading-order solver whose checks are bitmasks. The win is
the long tail: on the anti-brute board it is thousands of times fewer nodes.

In [18]:
for fn in (sudoku_naive, sudoku_rowmajor_bits, sudoku_mrv):
    (g, n), t = timed(fn, PUZZLES['Inkala'])
    print(f'Inkala  {fn.__name__:<22} {n:>7,} nodes  {t * 1000:6.0f} ms')

Inkala  sudoku_naive            49,559 nodes     297 ms
Inkala  sudoku_rowmajor_bits    49,559 nodes      40 ms
Inkala  sudoku_mrv              15,573 nodes      70 ms


## 9. LeetCode 78 — Subsets

In [19]:
class Solution78:
    """LeetCode 78 - Subsets."""

    def subsets(self, nums: list[int]) -> list[list[int]]:
        res, path = [], []

        def go(start: int) -> None:
            res.append(path[:])                  # the copy is the whole trick
            for i in range(start, len(nums)):
                path.append(nums[i])
                go(i + 1)
                path.pop()

        go(0)
        return res

In [20]:
print(Solution78().subsets([1, 2, 3]))

[[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]


## 10. LeetCode 39 — Combination Sum

Two knobs matter. The **start index** (`go(i, ...)`, never going back to earlier
candidates) is what stops `[2, 2, 3]` from also appearing as `[2, 3, 2]` and
`[3, 2, 2]`. **Sorting plus `break`** is the pruning: once a candidate is larger
than what remains, every later one is too.

In [21]:
class Solution39:
    """LeetCode 39 - Combination Sum."""

    def combinationSum(self, candidates: list[int], target: int) -> list[list[int]]:
        candidates = sorted(candidates)          # sorted, so we can stop early
        res, path = [], []

        def go(start: int, remaining: int) -> None:
            if remaining == 0:
                res.append(path[:])
                return
            for i in range(start, len(candidates)):
                x = candidates[i]
                if x > remaining:
                    break                        # every later x is bigger too
                path.append(x)
                go(i, remaining - x)             # i, not i + 1: x may repeat
                path.pop()

        go(0, target)
        return res


def combination_sum_nodes(candidates: list[int], target: int, *, start_index=True,
                          sort_and_break=True) -> tuple[list[list[int]], int]:
    """Instrumented LeetCode 39 so the two knobs can be switched off:
    start_index=False restarts every level at candidate 0 (so [2,2,3] and
    [3,2,2] are both produced); sort_and_break=False keeps trying numbers
    that are already too big."""
    cand = sorted(candidates) if sort_and_break else list(candidates)
    res, path = [], []
    nodes = 0

    def go(start: int, remaining: int) -> None:
        nonlocal nodes
        nodes += 1
        if remaining == 0:
            res.append(path[:])
            return
        if remaining < 0:
            return
        for i in range(start if start_index else 0, len(cand)):
            if sort_and_break and cand[i] > remaining:
                break
            path.append(cand[i])
            go(i, remaining - cand[i])
            path.pop()

    go(0, target)
    return res, nodes

In [22]:
print(Solution39().combinationSum([2, 3, 6, 7], 7))
for kw in ({}, {'start_index': False}, {'sort_and_break': False}):
    res, nodes = combination_sum_nodes([2, 3, 6, 7], 7, **kw)
    print(f'{str(kw):<28} {len(res)} answers, {nodes:>3} nodes  {res}')
a, n_a = combination_sum_nodes([2, 3, 5, 7, 11], 30)
b, n_b = combination_sum_nodes([2, 3, 5, 7, 11], 30, sort_and_break=False)
print(f'[2,3,5,7,11] -> 30: {len(a)} answers; {n_a:,} nodes with break, {n_b:,} without')

[[2, 2, 3], [7]]
{}                           2 answers,  10 nodes  [[2, 2, 3], [7]]
{'start_index': False}       4 answers,  13 nodes  [[2, 2, 3], [2, 3, 2], [3, 2, 2], [7]]
{'sort_and_break': False}    2 answers,  28 nodes  [[2, 2, 3], [7]]
[2,3,5,7,11] -> 30: 64 answers; 575 nodes with break, 1,239 without


## 11. LeetCode 51 / 52 — N-Queens and N-Queens II

In [23]:
class Solution51:
    """LeetCode 51 - N-Queens."""

    def solveNQueens(self, n: int) -> list[list[str]]:
        cols, diag, anti = set(), set(), set()
        board: list[int] = []
        out: list[list[str]] = []

        def go(r: int) -> None:
            if r == n:
                out.append(['.' * c + 'Q' + '.' * (n - c - 1) for c in board])
                return
            for c in range(n):
                if c in cols or r - c in diag or r + c in anti:
                    continue
                cols.add(c); diag.add(r - c); anti.add(r + c); board.append(c)
                go(r + 1)
                board.pop(); cols.remove(c); diag.remove(r - c); anti.remove(r + c)

        go(0)
        return out


class Solution52:
    """LeetCode 52 - N-Queens II (count only, bitmask)."""

    def totalNQueens(self, n: int) -> int:
        full = (1 << n) - 1

        def go(cols: int, ld: int, rd: int) -> int:
            if cols == full:
                return 1
            total = 0
            free = full & ~(cols | ld | rd)
            while free:
                bit = free & -free
                free ^= bit
                total += go(cols | bit, ((ld | bit) << 1) & full, (rd | bit) >> 1)
            return total

        return go(0, 0, 0)

In [24]:
print(Solution51().solveNQueens(4))
print([Solution52().totalNQueens(n) for n in range(1, 11)])

[['.Q..', '...Q', 'Q...', '..Q.'], ['..Q.', 'Q...', '...Q', '.Q..']]
[1, 0, 0, 2, 10, 4, 40, 92, 352, 724]


## 12. LeetCode 37 — Sudoku Solver (MRV + bitmasks, in place)

In [25]:
class Solution37:
    """LeetCode 37 - Sudoku Solver (in place, MRV + bitmasks)."""

    def solveSudoku(self, board: list[list[str]]) -> None:
        rows = [0] * 9; cols = [0] * 9; boxes = [0] * 9
        empty = []
        for r in range(9):
            for c in range(9):
                if board[r][c] == '.':
                    empty.append((r, c))
                else:
                    bit = 1 << int(board[r][c])
                    rows[r] |= bit; cols[c] |= bit; boxes[r // 3 * 3 + c // 3] |= bit

        def go() -> bool:
            if not empty:
                return True
            best, best_free, best_k = -1, 0, 10
            for i, (r, c) in enumerate(empty):
                free = ~(rows[r] | cols[c] | boxes[r // 3 * 3 + c // 3]) & 0x3FE
                k = bin(free).count('1')
                if k < best_k:
                    best, best_free, best_k = i, free, k
                    if k <= 1:
                        break
            if best_k == 0:
                return False
            r, c = empty[best]
            empty[best] = empty[-1]; empty.pop()
            b = r // 3 * 3 + c // 3
            while best_free:
                bit = best_free & -best_free
                best_free ^= bit
                rows[r] |= bit; cols[c] |= bit; boxes[b] |= bit
                board[r][c] = str(bit.bit_length() - 1)
                if go():
                    return True
                rows[r] ^= bit; cols[c] ^= bit; boxes[b] ^= bit
            board[r][c] = '.'
            empty.append((r, c))
            empty[best], empty[-1] = empty[-1], empty[best]
            return False

        go()

In [26]:
board = [list(PUZZLES['LeetCode 37'][r * 9:r * 9 + 9]) for r in range(9)]
Solution37().solveSudoku(board)
for row in board:
    print(' '.join(row))

5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9


## 13. LeetCode 79 — Word Search, and the missing restore

The board cell is marked `'#'` while it is on the current path and must be put
back on the way out. Delete that one line and all three of LeetCode's examples
still pass. Enumerate every string the board really spells and the broken
version misses nearly a fifth of them, starting with `CCB`.

In [27]:
class Solution79:
    """LeetCode 79 - Word Search."""

    def exist(self, board: list[list[str]], word: str) -> bool:
        R, C = len(board), len(board[0])

        def dfs(r: int, c: int, i: int) -> bool:
            if i == len(word):
                return True
            if not (0 <= r < R and 0 <= c < C) or board[r][c] != word[i]:
                return False
            saved, board[r][c] = board[r][c], '#'        # choose: mark as used
            found = (dfs(r + 1, c, i + 1) or dfs(r - 1, c, i + 1)
                     or dfs(r, c + 1, i + 1) or dfs(r, c - 1, i + 1))
            board[r][c] = saved                          # unchoose: ALWAYS
            return found

        return any(dfs(r, c, 0) for r in range(R) for c in range(C))


def exist_no_restore(board: list[list[str]], word: str) -> bool:
    """LeetCode 79 with the unchoose line deleted (works on a copy)."""
    g = [row[:] for row in board]
    R, C = len(g), len(g[0])

    def dfs(r: int, c: int, i: int) -> bool:
        if i == len(word):
            return True
        if not (0 <= r < R and 0 <= c < C) or g[r][c] != word[i]:
            return False
        g[r][c] = '#'                                   # BUG: never put back
        return (dfs(r + 1, c, i + 1) or dfs(r - 1, c, i + 1)
                or dfs(r, c + 1, i + 1) or dfs(r, c - 1, i + 1))

    return any(dfs(r, c, 0) for r in range(R) for c in range(C))


WORD_BOARD = [list('ABCE'), list('SFCS'), list('ADEE')]


def all_board_words(board: list[list[str]], max_len: int = 7) -> set[str]:
    """Every string spelled by a simple path of length <= max_len."""
    R, C = len(board), len(board[0])
    out: set[str] = set()

    def walk(r: int, c: int, seen: set, s: str) -> None:
        out.add(s)
        if len(s) == max_len:
            return
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            rr, cc = r + dr, c + dc
            if 0 <= rr < R and 0 <= cc < C and (rr, cc) not in seen:
                walk(rr, cc, seen | {(rr, cc)}, s + board[rr][cc])

    for r in range(R):
        for c in range(C):
            walk(r, c, {(r, c)}, board[r][c])
    return out

In [28]:
for w in ['ABCCED', 'SEE', 'ABCB', 'CCB']:
    print(f'{w:<7} correct: {Solution79().exist([r[:] for r in WORD_BOARD], w)!s:<6}'
          f' no restore: {exist_no_restore(WORD_BOARD, w)}')
words = all_board_words(WORD_BOARD)
missed = sorted((w for w in words if not exist_no_restore(WORD_BOARD, w)),
                key=lambda w: (len(w), w))
print(f'{len(words):,} real strings; broken version misses {len(missed)}; '
      f'shortest: {missed[:4]}')

ABCCED  correct: True   no restore: True
SEE     correct: True   no restore: True
ABCB    correct: False  no restore: False
CCB     correct: True   no restore: False
1,108 real strings; broken version misses 201; shortest: ['CCB', 'EEC', 'EED', 'ASAB']


## 14. Tests

In [29]:
assert sorted(map(tuple, subsets([1, 2, 3]))) == sorted(
    tuple(x for x, keep in zip([1, 2, 3], bits) if keep) for bits in product([0, 1], repeat=3))
assert len(bad) == 8 and all(x == [] for x in bad)
assert [len(queens(n)) for n in range(1, 11)] == [1, 0, 0, 2, 10, 4, 40, 92, 352, 724]
assert st['nodes'] == 2057 and queens_by_permutation(8)[0] == 92
assert queens_no_undo(8)[:2] == ([], 9)
assert all(queens_count_half_fixed(n) == len(queens(n)) for n in range(1, 11))
assert [queens_count_half(n) == len(queens(n)) for n in range(1, 11)] == [n % 2 == 0 or n == 3 for n in range(1, 11)]
assert s_sets['nodes'] == s_bits['nodes'] == 35_539
for name, s in PUZZLES.items():
    g, _ = sudoku_mrv(s)
    assert is_valid_solution(g, s)
assert sudoku_naive(PUZZLES['LeetCode 37'])[1] == 4209 and sudoku_mrv(PUZZLES['LeetCode 37'])[1] == 52
assert sorted(Solution39().combinationSum([2, 3, 6, 7], 7)) == [[2, 2, 3], [7]]
assert len(combination_sum_nodes([2, 3, 6, 7], 7, start_index=False)[0]) == 4
assert [Solution52().totalNQueens(n) for n in range(1, 9)] == [1, 0, 0, 2, 10, 4, 40, 92]
assert is_valid_solution([[int(x) for x in row] for row in board], PUZZLES['LeetCode 37'])
assert all(Solution79().exist([r[:] for r in WORD_BOARD], w) for w in words)
assert not exist_no_restore(WORD_BOARD, 'CCB') and len(missed) == 201
print('all assertions passed')

all assertions passed
